# Tree ensembles

A single decision tree is interpretable but unstable and easy to overfit. **Ensembles** combine
many trees: a random forest averages independent, decorrelated trees, and gradient boosting fits
trees in sequence, each correcting the last. This notebook compares both on the Adult income task.

## Learning objectives

By the end of this notebook you will be able to:

- explain bagging and boosting in one sentence each;
- fit a random forest and a gradient-boosted classifier;
- read feature importances and describe their bias toward high-cardinality features;
- compare ensembles with a single tree on the same split;
- reason about the cost of ensembles in training time and interpretability.

## Concept

**Bagging** trains many trees on bootstrap samples of the rows and averages their votes. Because
each tree sees a random subset of features at every split, the trees are decorrelated and their
errors partly cancel. A **random forest** is bagged trees. It reduces variance without adding much
bias, and it is hard to overfit by adding trees (though other hyperparameters still matter).

**Boosting** trains trees sequentially: each new tree fits the residuals (or the negative gradient)
of the current ensemble, so later trees focus on the examples earlier ones got wrong. Gradient
boosting is powerful and often the most accurate tabular model, but it can overfit if the learning
rate is high or there are too many trees, and training is sequential rather than parallel.

**Feature importances** from tree ensembles measure how much each feature reduced impurity across
splits. They are useful but biased: features with many distinct values (or one-hot columns from a
high-cardinality category) can look more important than they are. Permutation importance is a
fairer alternative.

## Worked example

### Prepare the Adult data

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
import ml
from ds_practice import load_uci_adult, set_seed, classification_metrics
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

set_seed(42)
adult = load_uci_adult().sample(6000, random_state=42).reset_index(drop=True)
adult = adult.assign(target=(adult["income"] == ">50K").astype(int))
X = adult.drop(columns=["income", "target"])
y = adult["target"]
numeric = X.select_dtypes("number").columns.tolist()
categorical = X.select_dtypes("object").columns.tolist()
preprocess = ColumnTransformer([
    ("num", "passthrough", numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("train/test:", len(X_train), len(X_test))

train/test: 4800 1200


### Single tree versus ensembles

We run a shallow tree as the baseline, then a forest and a gradient-boosted model. Each uses the same
preprocessing.

In [2]:
candidates = {
    "single_tree": ml.make_classifier("tree", max_depth=6),
    "random_forest": ml.make_classifier("forest", n_estimators=200),
    "gradient_boosting": ml.make_classifier("boosting", n_estimators=150, learning_rate=0.1),
}
rows = []
for name, model in candidates.items():
    pipeline = Pipeline([("prep", preprocess), ("model", model)])
    pipeline.fit(X_train, y_train)
    metrics = classification_metrics(y_test, pipeline.predict(X_test))
    rows.append({"model": name, **{k: round(v, 3) for k, v in metrics.items()}})
display(pd.DataFrame(rows))

,model,accuracy,precision,recall,f1
0,single_tree,0.856,0.854,0.856,0.842
1,random_forest,0.857,0.852,0.857,0.854
2,gradient_boosting,0.874,0.870,0.874,0.869


### Feature importances

The forest exposes `feature_importances_`. After the `ColumnTransformer`, one-hot columns are
expanded, so we take the encoded feature names back from the fitted transformer.

In [3]:
forest = Pipeline([("prep", preprocess), ("model", ml.make_classifier("forest", n_estimators=200))])
forest.fit(X_train, y_train)
names = forest.named_steps["prep"].get_feature_names_out()
importances = pd.Series(forest.named_steps["model"].named_steps["model"].feature_importances_, index=names)
print("top 10 features by impurity importance:")
display(importances.sort_values(ascending=False).head(10).round(4))

top 10 features by impurity importance:


num__age                                  0.1384
num__fnlwgt                               0.1277
num__capital_gain                         0.0922
num__hours_per_week                       0.0800
num__education_num                        0.0636
cat__marital_status_Married-civ-spouse    0.0632
cat__relationship_Husband                 0.0446
num__capital_loss                         0.0294
cat__marital_status_Never-married         0.0261
cat__occupation_Prof-specialty            0.0226
dtype: float64

### Permutation importance is fairer

`permutation_importance` shuffles one feature at a time and measures the drop in score, which does
not reward high cardinality the way impurity importance can.

In [4]:
from sklearn.inspection import permutation_importance

sample_X = X_test.sample(400, random_state=42)
sample_y = y_test.loc[sample_X.index]
result = permutation_importance(
    forest, sample_X, sample_y, n_repeats=5, random_state=42, scoring="accuracy"
)
perm = pd.Series(result.importances_mean, index=sample_X.columns).sort_values(ascending=False)
display(perm.head(8).round(4))

capital_gain      0.0565
marital_status    0.0110
hours_per_week    0.0045
capital_loss      0.0020
fnlwgt            0.0015
relationship      0.0005
occupation       -0.0000
native_country   -0.0005
dtype: float64

## Exercises

1. **Learning rate.** Sweep `learning_rate` over 0.01, 0.1, and 0.5 for the boosting model with a
   fixed 150 trees. Report test F1 and explain the trade-off with training time.
2. **More trees.** Increase the forest to 500 trees and report whether accuracy changes materially.
   Why does adding trees rarely hurt a forest?
3. **Importance disagreement.** Compare the top five features from impurity and permutation
   importance. Name one feature whose rank changes and suggest why.

## Limitations

Ensembles trade interpretability for accuracy: it is hard to explain a single prediction from a
forest of hundreds of trees. Training is slower, and gradient boosting is sequential, which limits
parallelism. Impurity importance overstates high-cardinality features, so permutation importance
should accompany it. The Adult sample is small and its `fnlwgt` column is a sampling weight we have
kept for convenience; a production pipeline would treat it carefully and tune the ensembles with
cross-validation.